# Course 7: Privacy-Preserving Machine Learning
## Session 9: Privacy for graph data

**Goal:** train an edge-level private GNN on a citation-like graph and plot accuracy against epsilon.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. What is being protected
Edge-level DP hides whether any one edge exists (for example, whether two patients were in contact). Node-level DP hides a whole node with all its edges, which is much harder. This notebook does edge-level privacy; the features and labels are treated as public.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from math import log, sqrt
rng = np.random.default_rng(0); torch.manual_seed(0)
k, n = 4, 800
y = np.repeat(np.arange(k), n // k)
A = np.triu(rng.random((n, n)) < np.where(y[:, None] == y[None, :], 0.06, 0.004), 1); A = (A | A.T).astype(np.float32)
X = np.eye(k)[y] @ rng.normal(size=(k, 8)) * 0.5 + rng.normal(size=(n, 8))          # weak signal in the features
X = X / np.linalg.norm(X, axis=1, keepdims=True)                                   # every row has length 1
idx = rng.permutation(n); train, test = idx[:200], idx[200:]
print("nodes", n, "| edges", int(A.sum() / 2), "| mean degree %.1f" % A.sum(1).mean())

### 2. Aggregation perturbation
The graph is used only once: to add up each node's neighbours' features. Removing one edge between i and j changes row i by one unit-length vector and row j by another, so the L2 sensitivity is `sqrt(2)`. Add Gaussian noise for that sensitivity and everything after it is post-processing. This is the core idea of GAP (Sajadmanesh et al. 2023), reduced to one hop.

In [ ]:
def gaussian_sigma(sensitivity, eps, delta):
    L = log(1 / delta); rho = (sqrt(L + eps) - sqrt(L)) ** 2      # zCDP: eps = rho + 2 sqrt(rho ln(1/delta))
    return sensitivity / sqrt(2 * rho)

def private_aggregate(eps, delta=1e-5, seed=0):
    agg = A @ X
    if eps is None: return agg
    sigma = gaussian_sigma(sqrt(2), eps, delta)
    return agg + np.random.default_rng(seed).normal(scale=sigma, size=agg.shape)

for eps in [1, 4, 16]: print(f"epsilon {eps:2d}: noise per number {gaussian_sigma(sqrt(2), eps, 1e-5):.2f}  (a typical aggregated row has length {np.linalg.norm(A @ X, axis=1).mean():.2f})")

### 3. Train a classifier on [own features, noisy aggregate]

In [ ]:
def run(agg, seed=0):
    torch.manual_seed(seed)
    feats = X if agg is None else np.c_[X, agg / np.sqrt(A.sum(1).mean())]
    Z = torch.tensor(feats, dtype=torch.float32); yt = torch.tensor(y)
    net = nn.Sequential(nn.Linear(Z.shape[1], 32), nn.ReLU(), nn.Linear(32, k))
    opt = torch.optim.Adam(net.parameters(), lr=0.01, weight_decay=1e-3)
    for _ in range(300):
        loss = F.cross_entropy(net(Z[train]), yt[train]); opt.zero_grad(); loss.backward(); opt.step()
    return (net(Z[test]).argmax(1) == yt[test]).float().mean().item()

import matplotlib.pyplot as plt
acc_none = run(None); acc_full = run(private_aggregate(None))
eps_grid = [0.5, 1, 2, 4, 8, 16, 32]
accs = [np.mean([run(private_aggregate(e, seed=s), seed=s) for s in range(3)]) for e in eps_grid]
print(f"no graph at all (perfect edge privacy)  accuracy {acc_none:.3f}")
for e, a in zip(eps_grid, accs): print(f"edge-level epsilon {e:4.1f}                 accuracy {a:.3f}")
print(f"graph with no privacy                   accuracy {acc_full:.3f}")
assert acc_full > acc_none and accs[-1] > accs[0]
plt.semilogx(eps_grid, accs, "o-"); plt.axhline(acc_none, ls="--"); plt.axhline(acc_full, ls=":")
plt.xlabel("edge-level epsilon"); plt.ylabel("test accuracy"); plt.savefig("graph_accuracy_vs_epsilon.png")

Read the curve against the two reference lines. Below some epsilon the noisy graph is no better than using no graph. An honest paper shows that 'no graph' baseline.

### 4. An alternative: randomized response on the edges
Flip every entry of the adjacency matrix with probability `1 / (1 + e^epsilon)`. It is simple and needs no trusted server, but at small epsilon the graph fills with false edges.

In [ ]:
def randomized_response(eps, seed=0):
    r = np.random.default_rng(seed); flip = np.triu(r.random((n, n)) < 1 / (1 + np.exp(eps)), 1); flip = flip | flip.T
    return np.abs(A - flip).astype(np.float32)
for eps in [1, 4, 8]:
    Ar = randomized_response(eps)
    print(f"epsilon {eps}: true edges {int(A.sum() / 2)}, edges after randomized response {int(Ar.sum() / 2)}")

### TODO
1. Two hops: the second aggregation touches the graph again, so the budget must be split between the hops. Implement it and compare with one hop at the same total epsilon.
2. Why is node-level privacy harder? What is the sensitivity of `A @ X` if one node with degree 50 is removed? What does GAP do about it?
3. Repeat on Cora with PyTorch Geometric (NOT RUN here) and report the same three lines.